In [1]:
import requests
import json
import numpy as np
import time
import pandas as pd

from src.bedrock import get_titan_response
from src.bedrock import get_bearer_token
from src.intent_classification import make_query_json
from typing import Dict, List

import warnings
warnings.filterwarnings('ignore')

In [2]:
get_bearer_token()

MAX_ERRORS = 10
OPENSEARCH_DOMAIN_FQDN = 'https://vpc-int-use1-opensearch-ml-c32nkeiaudrckcr2ep7jghefje.us-east-1.es.amazonaws.com'
HEADERS = {
    'Content-Type': 'application/json',
    'Accept-Encoding': 'gzip',
}

# INDEX_NAME = 'qna-data'
INDEX_NAME =  'search-data-titan-embed-v2_1024'

query_json_path = "../data/final-questions.json"
with open( query_json_path, "r" ) as file:
    test_query_data = json.load(file)

Bearer token set as BEARER_TOKEN_STR global variable


In [4]:
def evaluate_queries(query_data: List[Dict], 
                     opensearch_domain: str,
                     index_name: str,
                     headers: Dict,
                     top_k: int = 5,
                     model_id = "amazon.titan-embed-text-v2:0-pgo"
                    ) -> Dict[str, pd.DataFrame]:
    """
    Evaluates queries for keyword, semantic, and hybrid searches with different boost values
    """
    # Define search types and boost values
    # base_search_types = ['keyword', 'semantic']
    base_search_types = []

    boost_values = [1e-2]  # from 0.0001 to 10
    
    # Create search_types list with hybrid_boost variations
    search_types = base_search_types + [f'hybrid_boost_{boost}' for boost in boost_values]
    results = {stype: [] for stype in search_types}
    
    for query_dict in query_data:
        input_query = query_dict['question']
        target_id = query_dict['id']
        
        # Get embedding once for all search types
        start_embed = time.time()
        response_dict = get_titan_response(input_query, model_id=model_id)
        embed_vec = response_dict["body"]["embedding"]
        latency_embed = time.time() - start_embed
        
        # Run base search types (keyword and semantic)
        for search_type in base_search_types:
            start_time = time.time()
            query_json = make_query_json(input_query, embed_vec, top_k, search_type)
            response = requests.post(
                f"{opensearch_domain}/{index_name}/_search",
                headers=headers,
                json=query_json
            )
            
            latency = time.time() - start_time
            latency = latency + latency_embed
            
            result_dict = response.json()
            result_list = result_dict['hits']['hits'] + [{"_source":{"id": "NaN"},"_score":0}]*(top_k-len(result_dict['hits']['hits']))
            hit_id_list = [result_list[i]["_source"]["id"] for i in range(top_k)]
            top1_score = result_list[0]["_score"]
            top2_score = result_list[1]["_score"]
            
            results[search_type].append({
                'query': input_query,
                'target_id': target_id,
                'hit_ids': hit_id_list,
                'latency': latency,
                'top1_score': top1_score,
                'top2_score': top2_score,
                'recall@1': 1 if target_id in hit_id_list[:1] else 0,
                'recall@3': 1 if target_id in hit_id_list[:3] else 0,
                'recall@5': 1 if target_id in hit_id_list[:5] else 0
            })
        
        # Run hybrid searches with different boost values
        for boost in boost_values:
            search_type = f'hybrid_boost_{boost}'
            start_time = time.time()
            
            query_json = make_query_json(input_query, embed_vec, top_k, 'hybrid', boost)
            response = requests.post(
                f"{opensearch_domain}/{index_name}/_search",
                headers=headers,
                json=query_json
            )
            
            latency = time.time() - start_time
            latency = latency + latency_embed
            
            result_dict = response.json()
            result_list = result_dict['hits']['hits'] + [{"_source":{"id": "NaN"},"_score":0}]*(top_k-len(result_dict['hits']['hits']))
            hit_id_list = [result_list[i]["_source"]["id"] for i in range(top_k)]
            top1_score = result_list[0]["_score"]
            top2_score = result_list[1]["_score"]
            
            results[search_type].append({
                'query': input_query,
                'target_id': target_id,
                'hit_ids': hit_id_list,
                'latency': latency,
                'top1_score': top1_score,
                'top2_score': top2_score,
                'recall@1': 1 if target_id in hit_id_list[:1] else 0,
                'recall@3': 1 if target_id in hit_id_list[:3] else 0,
                'recall@5': 1 if target_id in hit_id_list[:5] else 0
            })
    
    return {stype: pd.DataFrame(data) for stype, data in results.items()}

In [5]:
def get_summary_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates summary metrics from results DataFrame
    """
    metrics = {
        'Recall@1': df['recall@1'].mean(),
        'Recall@3': df['recall@3'].mean(),
        'Recall@5': df['recall@5'].mean(),
        'Avg Latency': df['latency'].mean(),
        'P10 Latency': df['latency'].quantile(0.1),
        'P90 Latency': df['latency'].quantile(0.9),
        'P99 Latency': df['latency'].quantile(0.99),
        'Max Latency': df['latency'].max()
    }
    return pd.DataFrame([metrics]).T

In [6]:
results_dfs = evaluate_queries(
    query_data=test_query_data,
    opensearch_domain=OPENSEARCH_DOMAIN_FQDN,
    index_name=INDEX_NAME,
    headers=HEADERS,
    top_k=5,
)

In [7]:
summary_metrics = {}
for search_type, df in results_dfs.items():
    summary_metrics[search_type] = get_summary_metrics(df)

In [9]:
combined_metrics = pd.concat(summary_metrics, axis=1)
combined_metrics.columns = combined_metrics.columns.get_level_values(0)

In [10]:
combined_metrics

,hybrid_boost_0.01
Recall@1,0.745247
Recall@3,0.884845
Recall@5,0.932645
Avg Latency,0.669910
P10 Latency,0.406687
P90 Latency,0.888014
P99 Latency,1.265481
Max Latency,6.527854


In [17]:
combined_metrics.to_csv("../data/titan_v2_1024_eval_results_c7g_all_allsamples.csv")

In [16]:
results_dfs['hybrid_boost_0.01'].to_csv("../data/titan_v2_1024_results_c7g_all_allsamples.csv")